In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import *

In [2]:
if 0:
    filepath = '/media/ssd2/sleeplab_final_files/psg_airgo_10hz_002.h5'
    data, hdr, fs = read_in_routine(filepath)

In [3]:
files_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'
files_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Undiagnosed_Apnea'
files_dir = './'

save_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'
# save_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Undiagnosed_Apnea'


In [4]:
if 1:
    ### merge icu extraction tables:
    ss1 = pd.read_csv(os.path.join(files_dir, 'summary_subjects_icu_1.csv'))
    ss2 = pd.read_csv(os.path.join(files_dir, 'summary_subjects_icu_2.csv'))
#     ss3 = pd.read_csv(os.path.join(files_dir, 'summary_subjects_icu_3.csv'))

    sd1 = pd.read_csv(os.path.join(files_dir, 'summary_days_icu_1.csv'))
    sd2 = pd.read_csv(os.path.join(files_dir, 'summary_days_icu_2.csv'))
#     sd3 = pd.read_csv(os.path.join(files_dir, 'summary_days_icu_3.csv'))


    ss = pd.concat([ss1, ss2], axis=0, sort=False) # , ss3
    ss = ss.sort_values(by=['study_id'])
    print(ss.shape)
    ss.to_csv(os.path.join(save_dir, 'summary_subjects_icu.csv'), index=False)
    

    sd = pd.concat([sd1, sd2], axis=0, sort=False) # , sd3
    sd = sd.sort_values(by=['study_id', 'day_no'])
    sd.to_csv(os.path.join(save_dir, 'summary_days_icu.csv'), index=False)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (14,90,91,92,6454,6455,6456,12818,12819,12820,19184,19185,19186,25548,25549,25550,31912,31913,31914,38278,38279,38280,44642,44643,44644,46850,46851,46852,46864,46865,46866,46878,46879,46880,46948,46949,46950,46955,46956,46957,46962,46963,46964,46969,46970,46971,46976,46977,46978,46983,46984,46985,51006,51007,51008,53335,53336,53337,57304,57372,57373,57374,59587,59588,59589,59601,59602,59603,59685,59686,59687,63668,63736,63737,63738,65951,65952,65953,65958,65959,65960,65979,65980,65981,66035,66036,66037,66042,66043,66044,66049,66050,66051,66056,66057,66058,66070,66071,66072,66077,66078,66079,70032,70100,70101,70102,72317,72318,72319,72331,72332,72333,72401,72402,72403,72415,72416,72417,76398,76466,76467,76468,78681,78682,78683,78695,78696,78697,78709,78710,78711,78765,78766,78767,78786,78787,78788,78800,78801,78802,78807,78808,78809,82762,82830,82831,82832,85

(108, 171868)


In [5]:
if 1:
    
    # add inclusion criteria for ICU (sleeplab fulfills all min data criteria and age is accounted for in matching.)
    
    import re
    # add total amount of data availability

    for col in ['airgo_available_h', 'ecg_available', 'airgo_ecg_available']:
        cols = [x for x in ss.columns if re.match(f'f\d_{col}', x) is not None]
        ss[col + '_sum'] = ss[cols].sum(axis=1)
        
        
    min_hours_available = 2
    min_age = 45
    # min_data_columns = [f'f{i}_airgo_available_h' for i in range(1,9)]
    # min_sleep_columns = [f'f{i}_sleep_hours_breathing' for i in range(1,9)]

    print(f"N Patients Start: {ss.shape[0]}")

    data_subjects_inclusion = ss.loc[ss.age >= min_age, :]
    print(f"N Patients After Age exclusion: {data_subjects_inclusion.shape[0]}")

    data_subjects_inclusion = data_subjects_inclusion.loc[data_subjects_inclusion.airgo_ecg_available_sum >= min_hours_available, :]
    print(f"N Patients After Little Data Available exclusion: {data_subjects_inclusion.shape[0]}")
    
    ss['inclusion_subject'] = np.isin(ss.study_id, data_subjects_inclusion.study_id).astype(int)
    sd['inclusion_subject'] = np.isin(sd.study_id, data_subjects_inclusion.study_id).astype(int)
    
    ss = ss[list(ss.columns[-1:]) + list(ss.columns[:-1])]
    sd = sd[list(sd.columns[-1:]) + list(sd.columns[:-1])]

    ss.to_csv(os.path.join(save_dir, 'summary_subjects_icu.csv'), index=False)
    sd.to_csv(os.path.join(save_dir, 'summary_days_icu.csv'), index=False)

N Patients Start: 108
N Patients After Age exclusion: 106
N Patients After Little Data Available exclusion: 102


### sleeplab: add some demographics and matching cols from old table to new table:

In [6]:
ss_sleeplab_old = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/summary_subjects_sleeplab_untilOct23.csv')

ss_sleeplab_new = pd.read_csv('summary_days_sleeplab.csv')

subjects_sleeplab_new = pd.read_csv('summary_subjects_sleeplab.csv')

for col in ['airgo_available_h', 'ecg_available', 'airgo_ecg_available']:
    cols = [x for x in subjects_sleeplab_new.columns if re.match(f'n\d_{col}', x) is not None]
    subjects_sleeplab_new[col + '_sum'] = subjects_sleeplab_new[cols].sum(axis=1)


In [7]:
# demographic:
ss_sleeplab_old = ss_sleeplab_old[['study_id', 'mrn', 'age', 'sex', 'cci']]
ss_sleeplab_old = ss_sleeplab_old.set_index('study_id')
ss_sleeplab_new = ss_sleeplab_new.set_index('study_id')
ss_sleeplab_new = ss_sleeplab_new.join(ss_sleeplab_old)
# ss_sleeplab_new = ss_sleeplab_new.set_index('study_id')

subjects_sleeplab_new = subjects_sleeplab_new.set_index('study_id')
subjects_sleeplab_new = subjects_sleeplab_new.join(ss_sleeplab_old)


In [8]:
# matching:
ss_sleeplab_old = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/summary_days_sleeplab_untilOct23.csv')
ss_sleeplab_old = ss_sleeplab_old.set_index('study_id')
match_cols = [x for x in ss_sleeplab_old.columns if 'matched' in x]
ss_sleeplab_new = ss_sleeplab_new.join(ss_sleeplab_old[match_cols])
ss_sleeplab_new[match_cols] = ss_sleeplab_new[match_cols].fillna(value=0)
ss_sleeplab_new = ss_sleeplab_new[list(ss_sleeplab_new.columns[-10:]) + list(ss_sleeplab_new.columns[:-10])]
ss_sleeplab_new.reset_index(inplace=True)

subjects_sleeplab_new = subjects_sleeplab_new.join(ss_sleeplab_old[match_cols])
subjects_sleeplab_new[match_cols] = subjects_sleeplab_new[match_cols].fillna(value=0)
subjects_sleeplab_new = subjects_sleeplab_new[list(subjects_sleeplab_new.columns[-10:]) + list(subjects_sleeplab_new.columns[:-10])]
subjects_sleeplab_new.reset_index(inplace=True)


In [9]:
ss_sleeplab_new.to_csv(os.path.join(save_dir, 'summary_days_sleeplab.csv'), index=False)
subjects_sleeplab_new.to_csv(os.path.join(save_dir, 'summary_subjects_sleeplab.csv'), index=False)

## add agreement, discordant sleep etc. to summary tables:

In [11]:
from icu_sleep_breathing_2020_help_functions import * 
import pickle
import itertools
from sleep_analysis_functions import compute_sleep_indices

if 1:
    
    [summary_subjects_icu, summary_subjects_sleeplab, 
     summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(apply_inclusion_criteria=False)
    
    # LHL data:
    data = pickle.load( open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_expert_label.p", "rb" ) )
    
    for population in ['sleeplab', 'icu']:

        if population == 'icu':
            summary_days = summary_days_icu.copy()
        elif population == 'sleeplab':
            summary_days = summary_days_sleeplab.copy()


        fs = 1/30 # hlh data in 30 seconds epoch.
        stages = ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort', 'stage_pred_expert']
        agreements = ['all', 'agreement', 'disagreement', 'agreement_relaxed', 'disagreement_relaxed']


        for jloc, row in summary_days.iterrows():
            statistics_result = pd.DataFrame()

            start_dt = row.timerange.split(' - ')[0]
            end_dt = row.timerange.split(' - ')[1]

            data_subject = data.loc[(data.population==population) & (data.study_id==row.study_id)]
            data_timerange = data_subject.loc[(data_subject.datetime >= start_dt) & (data_subject.datetime <= end_dt)]

            for stage, agreement in itertools.product(stages, agreements):
                stage_short = stage.replace('stage_pred_', '')
                name = stage_short + '_' + agreement

                if agreement == 'all':
                    data_sel = data_timerange
                elif agreement == 'agreement':
                    data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement == 1]
                elif agreement == 'disagreement':
                    data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement == 0]
                elif agreement == 'agreement_relaxed':
                    data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement_relaxed == 1]
                elif agreement == 'disagreement_relaxed':
                    data_sel = data_timerange[data_timerange.ecg_nn_amendsumeffort_agreement_relaxed == 0]

                statistics_result_tmp = compute_sleep_indices(data_sel, stage, fs, name=name)
                statistics_result = pd.concat([statistics_result, statistics_result_tmp], axis=1)

                statistics_result_tmp.index += '_' + name

                for new_col in list(statistics_result_tmp.index):
                    if not new_col in summary_days.columns:
                        summary_days[new_col] = np.nan

                summary_days.loc[jloc, list(statistics_result_tmp.index)] = statistics_result_tmp.values.flatten()

        if population == 'icu':
            summary_days_icu = summary_days.copy()
        elif population == 'sleeplab':
            summary_days_sleeplab = summary_days.copy()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3254: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,78

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
24-hour segments: 410
12-hour segments: 820

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [12]:
if 1:
    summary_days_icu.to_csv(os.path.join(save_dir, 'summary_days_icu.csv'), index=False)
    summary_days_sleeplab.to_csv(os.path.join(save_dir, 'summary_days_sleeplab.csv'), index=False)

In [10]:
if 0:
    
    files_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'
    
    summary_subjects = pd.read_csv(os.path.join(files_dir, 'summary_subjects_icu.csv'))
    summary_subjects.columns = summary_subjects.columns.str.strip()
#     summary_subjects = summary_subjects.round(1)

    summary_days = pd.read_csv(os.path.join(files_dir, 'summary_days_icu.csv'))
    summary_days.columns = summary_days.columns.str.strip()
#     summary_days = summary_days.round(1)

    summary_days['med_surg'] = summary_days['med_surg'].apply(lambda x: x.strip())
    summary_days['med_surg'] = (summary_days['med_surg'] == 'Medical').astype(int)
    summary_subjects['med_surg'] = summary_subjects['med_surg'].apply(lambda x: x.strip())
    summary_subjects['med_surg'] = (summary_subjects['med_surg'] == 'Medical').astype(int)

    for summary_df in [summary_subjects, summary_days]:

        summary_df['copd'] = summary_df['copd'].apply(lambda x: x.strip())
        summary_df['copd'] = (summary_df['copd'] == 'yes').astype(int)

        summary_df['chf'] = summary_df['chf'].apply(lambda x: x.strip())
        summary_df['chf'] = (summary_df['chf'] == 'yes').astype(int)

    

In [13]:
if 1:
    
    import re

    study_ids_airgo_good = summary_subjects.study_id.values
    print(f'Number of subjects with airgo available and sleep detected: {len(study_ids_airgo_good)}')


    icu_with_ecg_done_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step4'
    study_ids_ecg_done = [int(re.search('\d\d\d', x)[0]) for x in os.listdir(icu_with_ecg_done_dir)]

    print(f'Number of subjects with ecg sleep staging done: {len(study_ids_ecg_done)}')

    print(f'Intersection of above: {len(list( set(study_ids_airgo_good).intersection(set(study_ids_ecg_done))))}')
    print(f'ECG good but not in summary table: {list( set(study_ids_ecg_done) - set(study_ids_airgo_good))}')


Number of subjects with airgo available and sleep detected: 108
Number of subjects with ecg sleep staging done: 109
Intersection of above: 108
ECG good but not in summary table: [42]


In [14]:
if 1:
    summary_subjects.to_csv(os.path.join(files_dir, 'summary_subjects_icu.csv'), index=False)
    summary_days.to_csv(os.path.join(files_dir, 'summary_days_icu.csv'), index=False)


In [7]:
# merge sleeplab summary tables with demographics data

In [139]:
dir_summaries = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing'
summary_subjects_sleeplab = pd.read_csv(os.path.join(dir_summaries, 'summary_subjects_sleeplab.csv'))
summary_days_sleeplab = pd.read_csv(os.path.join(dir_summaries, 'summary_days_sleeplab.csv'))

In [141]:
summary_subjects_sleeplab.head(2)

,study_id,mrn,age,sex,cci,n1_timerange,n1_hr_mean,n1_hr_std,n1_hr_median,n1_hr_q25,...,n1_ventilation_cvar_30sec_q75,n1_ventilation_cvar_30sec_sda,n1_ventilation_cvar_30sec_asd,n1_inht_cycle_ratio_10sec_mean,n1_inht_cycle_ratio_10sec_std,n1_inht_cycle_ratio_10sec_median,n1_inht_cycle_ratio_10sec_q25,n1_inht_cycle_ratio_10sec_q75,n1_inht_cycle_ratio_10sec_sda,n1_inht_cycle_ratio_10sec_asd
0,1,6027967.0,31.085,Male,0.0,2019-01-23 20:00:00 - 2019-01-24 08:00:00,61.348896,10.002581,61.0,59.0,...,0.107361,0.050253,0.073158,0.316239,0.050262,0.311279,0.286377,0.338135,0.020278,0.041499
1,2,6234275.0,27.789,Female,1.0,2019-01-23 20:00:00 - 2019-01-24 08:00:00,71.955284,5.198903,71.0,69.0,...,0.214111,0.110350,0.136028,0.331117,0.076752,0.327637,0.290039,0.367676,0.024191,0.067440


In [130]:
sleeplab_demographics = pd.read_csv('sleeplab_airgo_patients_demographics.csv')

In [131]:
sleeplab_demographics.head(3)

,MRN,Age,Sex,SID
0,183939,81.893,Male,Air303
1,405144,78.630,Male,Air087
2,611696,101.337,Female,Air196


In [132]:
# MULTIPLE STUDY NIGHTS:
masterlist = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleeplab/MasterList_Main_1.23.20.csv'
masterlist = pd.read_csv(masterlist)

study_ids_same_patient = {}

for mrn_multiple in mrns_multiple_studies:
    
    match = masterlist.loc[np.isin(masterlist.MRN.values, mrn_multiple)]
    sids = match.SID.values
    if any(pd.isna(sids)): continue
    study_ids_same_patient[sids[1]] = sids[0]
    study_ids_same_patient[sids[0]] = sids[1]


In [133]:
for jloc, row in summary_days_sleeplab.iterrows():
    study_id = str(row.study_id).zfill(3)
    
    demographics = sleeplab_demographics.loc[sleeplab_demographics.SID == 'Air'+study_id]
    if demographics.shape[0] == 0:
        study_id = study_ids_same_patient['Air'+study_id]
        demographics = sleeplab_demographics.loc[sleeplab_demographics.SID == study_id]

    summary_days_sleeplab.loc[jloc, 'mrn'] = demographics.MRN.values[0]
    summary_days_sleeplab.loc[jloc, 'age'] = demographics.Age.values[0]
    summary_days_sleeplab.loc[jloc, 'sex'] = demographics.Sex.values[0]
    
    summary_subjects_sleeplab.loc[jloc, 'mrn'] = demographics.MRN.values[0]
    summary_subjects_sleeplab.loc[jloc, 'age'] = demographics.Age.values[0]
    summary_subjects_sleeplab.loc[jloc, 'sex'] = demographics.Sex.values[0]

In [134]:
# CCI:
cci_table = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/CCI/CCI_edw_icd_codes_sleeplab_airgo_study.csv'
cci_table = pd.read_csv(cci_table)

In [135]:
print(np.all(np.isin(summary_days_sleeplab.mrn.unique(), cci_table.PatientIdentityID)))

for jloc, row in summary_days_sleeplab.iterrows():
    mrn = row.mrn
    cci = cci_table.loc[cci_table.PatientIdentityID == mrn, 'score'].values[0]
    summary_days_sleeplab.loc[jloc, 'cci'] = cci
    summary_subjects_sleeplab.loc[jloc, 'cci'] = cci

True


In [136]:
ordered_cols = []
ordered_cols += [summary_days_sleeplab.columns[0]]
ordered_cols.extend(summary_days_sleeplab.columns[-4:])
ordered_cols.extend(summary_days_sleeplab.columns[1:-4])
assert len(ordered_cols) == len(summary_days_sleeplab.columns)
summary_days_sleeplab = summary_days_sleeplab[ordered_cols]

ordered_cols = []
ordered_cols += [summary_subjects_sleeplab.columns[0]]
ordered_cols.extend(summary_subjects_sleeplab.columns[-4:])
ordered_cols.extend(summary_subjects_sleeplab.columns[1:-4])
assert len(ordered_cols) == len(summary_subjects_sleeplab.columns)
summary_subjects_sleeplab = summary_subjects_sleeplab[ordered_cols]

In [126]:
summary_subjects_sleeplab.to_csv(os.path.join(dir_summaries, 'summary_subjects_sleeplab.csv'), index=False)
summary_days_sleeplab.to_csv(os.path.join(dir_summaries, 'summary_days_sleeplab.csv'), index=False)